In [ ]:
import numpy as np
# Patch for NumPy >= 1.24 where np.bool8 was removed
if not hasattr(np, "bool8"):
    np.bool8 = np.bool_
if not hasattr(np, "np.float_"):
    np.float_ = np.float64
if not hasattr(np, "np.complex_"):
    np.complex_ = np.complex128
if not hasattr(np, "np.unicode_"):
    np.unicode_ = np.str_

# Patch for printing to log from .ipynb
import logging
def recbole_reset_logs():
    logging.shutdown()
    for handler in logging.root.handlers[:]:
        handler.close()
        logging.root.removeHandler(handler)

# Check for GPU availability
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("cuda version:", torch.version.cuda)
print("device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("gpu name:", torch.cuda.get_device_name(0))

In [ ]:
recbole_reset_logs()

In [ ]:
# import inspect, recbole.model.general_recommender.itemknn as ik
# print(inspect.getfile(ik))


# import numpy as np
# from recbole.config import Config
# from recbole.data import create_dataset
# config = Config(
#     model='ItemKNN',
#     dataset='ml-100k',
# )
# dataset = create_dataset(config)
# model = ik.ItemKNN(config, dataset)
# print(np.unique(model.interaction_matrix.data)[:10])


# # self.interaction_matrix = dataset.inter_matrix(form="csr").astype(np.float32)
# ##########
# self.interaction_matrix = dataset.inter_matrix(form="csr", value_field=config["RATING_FIELD"]).astype(np.float32)
# ##########

In [ ]:
from recbole.quick_start import run_recbole

models = ['ItemKNN', 'BPR', 'NeuMF', 'GCMC']
datasets = ['ml-100k', 'jester', 'amazon_all_beauty', 'douban', 'netflix']
datatypes = ['expR', 'impR', 'impB']

MovieLens, Jester, Amazon, Douban, Netflix = range(5)
ItemKNN, BPR, NeuMF, GCMC = range(4)
expR, impR, impB = range(3)

for m in [ItemKNN, BPR, NeuMF, GCMC]:
    for d in [Amazon, MovieLens, Douban, Jester]:      
        for t in [impB, impR, expR]:      
            model = models[m]            
            dataset = datasets[d]       
            datatype = datatypes[t]

            config_dict = {
                # --- Model + data selection ---
                'model': model,
                'data_path': f'./data/split/{dataset}',                                
                'dataset': f'{dataset}--{datatype}',      
                'benchmark_filename': ['train','valid','test'],                    

                # --- Use GPU ---
                'use_gpu': True,
                'num_workers': 4,
                'pin_memory': True,
                
                # --- ---
                'show_progress': False,

                #--- ---
                'load_col': {'inter': ['user_id', 'item_id', 'rating']},
                'RATING_FIELD': 'rating', 

                'eval_args': {
                    'split': {'RS': [0, 0, 0]}, # Disable RecBole auto-splitting
                    'mode': 'uni100'},
                'test_args': {'mode': 'full'},

                # --- Evaluation targets ---
                'topk': [10, 20],               # Top-k recommendations to consider
                'metrics': ['Precision', 'Recall', 'MRR', 'Hit', 'NDCG'],

                # --- Training schedule + early stopping ---
                'epochs': 50,                   # Max epochs
                'eval_step': 1,                 # Evaluate every epoch
                'stopping_step': 5,             # Stop if no improvement for 5 evals (i.e., 5 epochs)
                'valid_metric': 'NDCG@10',      # Choose ONE metric for early stopping
                'valid_metric_bigger': True     # because higher NDCG is better
            }
            
            recbole_reset_logs()
            run_recbole(config_dict=config_dict)
recbole_reset_logs()